# 09_00 Advanced: answers grounded in Homer's text (overview)

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: real questions, answered from the real text
>
> Project files this connects to:
>
> - `src/homer_ai/retrieval.py`
> - `src/homer_ai/homer_mcp_server.py`
> - `data/eval/questions.jsonl`
> - `scripts/benchmark_grounding.py`
>
> **This notebook:** Ask a model about Homer and it answers from memory, with nothing you can check. How do we get an answer that comes from the real text, with the book and line beside it?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a grounded answer is, and why it needs a search step before the model
- Why our own Homer model can't answer a question, and what it can still add
- The parts we build: passages, a word search called BM25, a way to score the search, and an MCP tool
- What results to expect, including the ones that surprised me

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **retrieval** | finding the pieces of text that fit a question, before anybody answers it | the 5 paragraphs of Homer that best match your question |
| **grounded answer** | an answer written from text that was put in front of the model, not from the model's memory | Claude answers from 5 passages we handed it |
| **citation** | the place an answer came from, written so you can look it up | `odyssey 9:24`: the Odyssey, Book 9, line 24 of our file |
| **passage** | a short piece of a text that is searched and shown as one unit; also called a chunk | one paragraph of the English, or 5 verses of the Greek |
| **index** | the tables a search prepares once, so that each question is answered fast | for every word, the passages that contain it |

## 3. The idea

### ❓ What are we building?

You ask a question about Homer in English, and you get an answer that comes from the real text, with the place written next to it: *Odyssey, Book 9, line 24*. So you can open the book and check.

Here's the picture I use. Think of a closed-book exam and an open-book exam.

In a closed-book exam you answer from memory. That's what a language model does when you just ask it. It may be right, but you can't see where the answer came from, and when its memory is wrong it sounds exactly as sure.

In an open-book exam, a librarian first fetches the few pages that fit the question. You read them, you answer, and you write the page number beside your answer. That's this chapter. The librarian is a search program we write ourselves. The student is Claude. The page number is the **citation**.

| In the picture | In this chapter | Notebook |
|---|---|---|
| the book, cut into pages that know their number | both poems as **passages**, each with its book and line | [09_01](09_01_chunking_and_bm25.ipynb) |
| the librarian who fetches the right pages | a word search called **BM25**, built from scratch | [09_01](09_01_chunking_and_bm25.ipynb), [09_02](09_02_search_and_scores.ipynb) |
| checking how often the librarian brings the right page | 40 labelled questions, **recall@k** and **MRR** | [09_02](09_02_search_and_scores.ipynb) |
| a desk where any student can ask the librarian | the MCP tool `search_homer` | [09_03](09_03_the_search_tool.ipynb) |
| the exam, with the book open and closed | Claude with and without the passages | [09_04](09_04_with_and_without.ipynb) |

The picture has a limit. Our librarian can't read. It matches words: ask about *pigs* and it finds pages with the word *pigs*. A page that says the same thing in other words stays on the shelf. You'll see exactly how often that happens, because we measure it.

The usual name for all of this is **RAG**, retrieval-augmented generation ([08_00](../08_advanced_fine_tuning/08_00_chapter_overview.ipynb) put it on step 2 of the ladder). The next cell draws it.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("Grounded answers", "Fetch the page first, then answer", (10, 6.6))
ax = diagram_axes(fig)

box(ax, 1.6, 4.9, 2.8, 0.9, "a question\nin English", INK, fontsize=10)
box(ax, 5.0, 4.9, 2.6, 0.9, "search\n(BM25)", ACCENT, fontsize=11, lw=2.2)
box(ax, 8.4, 4.9, 2.8, 0.9, "the 5 best\npassages", GOOD, fontsize=10)
arrow(ax, (3.0, 4.9), (3.7, 4.9), color=MUTED, width=2)
arrow(ax, (6.3, 4.9), (7.0, 4.9), color=GOOD, width=2)
tag(ax, 5.0, 5.65, "1. retrieve", ACCENT, 10, weight="bold")

box(ax, 5.0, 3.2, 4.4, 0.75, "both poems, cut into 2,140 passages", MUTED,
    fontsize=9.5, dashed=True)
arrow(ax, (5.0, 3.6), (5.0, 4.4), color=MUTED)
tag(ax, 5.0, 2.62, "each one remembers its place: odyssey 9:24", MUTED, 9, mono=True)

box(ax, 8.4, 1.3, 2.8, 1.0, "Claude reads them\nand answers", INK, fontsize=10)
arrow(ax, (8.4, 4.4), (8.4, 1.85), color=GOOD, width=2)
arrow(ax, (1.6, 4.4), (1.6, 1.3), color=MUTED, dashed=True)
arrow(ax, (1.6, 1.3), (2.75, 1.3), color=MUTED, dashed=True)
tag(ax, 0.85, 2.9, "the question\ngoes along", MUTED, 8.5)
tag(ax, 8.4, 0.45, "2. answer", ACCENT, 10, weight="bold")
box(ax, 4.6, 1.3, 3.6, 1.0, '"Noman."\n[odyssey 9:24]', GOOD, mono=True, fontsize=10.5,
    lw=2.2)
arrow(ax, (7.0, 1.3), (6.45, 1.3), color=GOOD, width=2)
tag(ax, 4.6, 0.45, "an answer you can check", GOOD, 9)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two steps. Step 1, **retrieve**: the question goes to the search, which picks the 5 best of the 2,140 English passages. Each passage carries its place, like `odyssey 9:24`. Step 2, **answer**: Claude gets the question and those 5 passages, and writes an answer with the place it used. No model is trained anywhere in this chapter.

</details>

### ❓ We built a Homer model. Why doesn't it answer the question?

Because it can't, and it's better to see that than to take my word for it.

Our GPT reads one character at a time, and it learned from Book 1 of the Odyssey in Greek. So it knows which letter tends to follow which. It has no idea what a question is, and it has never seen most English letters.

👉 **Predict first:** we give it `Who blinded the Cyclops?`. How many of those 24 characters does it know?

In [ ]:
import torch

from homer_ai.generation import next_scores
from homer_ai.lessons import load_homer_gpt

tokenizer, model = load_homer_gpt()  # read-only
question = "Who blinded the Cyclops?"
ids = tokenizer.encode(question)
unknown = sum(i == tokenizer.unk_id for i in ids)
print(f"{len(ids)} characters, {unknown} of them unknown to the model")
print("what it reads:", tokenizer.decode(ids))

generated = list(ids)
with torch.no_grad():  # just guessing, not learning
    for _ in range(40):
        generated.append(torch.argmax(next_scores(model, generated)).item())
print("what it adds: ", repr(tokenizer.decode(generated[len(ids) :])))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

It knows 3 of the 24 characters: the three spaces. Every letter of the question, and the question mark, is `<|unk|>` to it, because Book 1 has no Latin letters. Then it does the only thing it can: it adds Greek letters that look like Homer. That isn't an answer to anything.

A bigger model of the same kind would still have the first problem of the closed-book exam: no page number.

</details>

### ❓ So is our model out of the picture?

Almost. It keeps one small job. Our MCP server already has the tool `consult_homer_llm` from [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb), which continues a Greek seed in Homer's style. This chapter adds a second tool to the same server, `search_homer`. So Claude can fetch the real passage with one tool, and, if you ask for it, get a few made-up Greek letters in Homer's style from the other.

The answer itself always comes from the real text. The model's Greek is decoration, and the tool's description says so.

### ❓ Does search mean embeddings and a vector database?

Often, yes. Not here, and that's a choice.

We start with a search that counts words: **BM25**. It's what search engines used for decades, it fits in about 40 lines of plain Python, and you can check every number by hand. In [09_02](09_02_search_and_scores.ipynb) we compare it, on the same questions, with a small pretrained model that turns sentences into vectors. I expected the vectors to win easily. They didn't.

We also try the vectors we already have, the token embeddings of our own model. They fail completely, and it's worth seeing why.

### ❓ What do I need on my computer?

Nothing new. No download, no training, and every notebook of this chapter runs in a few seconds.

- The text is already in `data/homer/`: both poems, in Greek and in Samuel Butler's English.
- The finished code lives in `src/homer_ai/retrieval.py`. It uses only Python's standard library, so there's nothing new to install.
- One cell in 09_02 is optional. It needs the `sentence-transformers` library and a model of about 90 MB. Without them it prints one line and moves on. To try it: `uv sync --extra rag`.
- [09_04](09_04_with_and_without.ipynb) describes a script that calls Claude. You don't have to run it. The notebook shows one recorded run.

### ❓ How good will the results be?

Here's what I measured, on 40 questions I wrote and labelled by hand:

| Notebook | What we measure | Result |
|---|---|---|
| 09_02 | is a right passage among the first 5 the search returns? | 22 of 40 questions (0.55) |
| 09_02 | the same, with a pretrained sentence model in place of BM25 | 19 of 40 (0.475) |
| 09_02 | the same, with our own model's letter vectors | useless: a word and its scrambled letters get the same vector |
| 09_04 | Claude names the right book, without and with our passages (8 questions) | 8 of 8 without, 5 of 8 with |

Read the last row twice. Claude has read Homer, so on a famous poem the closed-book exam goes very well, and our half-good search can even hold it back. What the open book adds is a line you can check, and an honest *the passages don't say* when the search missed. On text a model has never read, like your own documents, there is no closed-book option at all. 09_04 goes through it.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`09_01_chunking_and_bm25.ipynb`](09_01_chunking_and_bm25.ipynb) | How do I cut a text into passages, and how does a word search score them? |
| [`09_02_search_and_scores.ipynb`](09_02_search_and_scores.ipynb) | How good is the search, measured on real questions, and would vectors do better? |
| [`09_03_the_search_tool.ipynb`](09_03_the_search_tool.ipynb) | How do Claude and other AI apps get to use the search? |
| [`09_04_with_and_without.ipynb`](09_04_with_and_without.ipynb) | Does it help? And what would this look like on a company's own documents? |

## 4. Recap

- A grounded answer is written from text that a search step fetched, and it says where the text is.
- Our character-level model can't answer questions. It doesn't even know the letters of an English question.
- The search is BM25: word counting, built from scratch, no new library.
- We measure the search on 40 labelled questions before we trust it, and we report the numbers as they came.

**Next:** [09_01: Passages, and a word search from scratch](09_01_chunking_and_bm25.ipynb).